# DeepLab old baseline reproduction on `kurgans-dataset/segmentation_dataset`

Этот notebook запускает old-style recipe из `all_class_baseline.ipynb`, но через clean scripts модуля `03_multiclass_segmentation_deeplab`.

Цель: проверить, воспроизводится ли сильная 5-class DeepLab линия на текущем Kaggle dataset:

`/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset/segmentation_dataset`

Recipe, который воспроизводим:

- DeepLabV3+ / ResNet34
- 6 classes: background + 5 archaeology classes
- metadata filtering: `max_crop_size=2048`, `max_objects_in_patch=40`, `touches_border=False`
- manual val regions from `all_class_baseline.ipynb`
- CE + Dice: `ce_weight=0.7`, `dice_weight=0.3`
- class weights: `[0.2, 1.0, 1.0, 1.4, 1.8, 1.8]`
- Adam, `lr=1e-4`
- ReduceLROnPlateau
- early stopping patience 12
- best checkpoint by `val_mean_fg_iou`

Notebook не содержит training logic: он только вызывает `scripts/train.py`, `scripts/evaluate.py`, `scripts/visualize_predictions.py`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import torch

print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## Clone or update repository

In [ ]:
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/MataNerdy/Geodata_Archaeology_CV.git')
BRANCH = os.environ.get('BRANCH', 'main')
REPO_DIR = Path('/kaggle/working/Geodata_Archaeology_CV')

if REPO_DIR.exists():
    print('Repo already exists. Pulling latest changes...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

SEG_DIR = REPO_DIR / '03_multiclass_segmentation_deeplab'
assert SEG_DIR.exists(), SEG_DIR
os.environ['PYTHONPATH'] = str(SEG_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Using SEG_DIR:', SEG_DIR)
subprocess.run(['git', 'log', '--oneline', '-3'], cwd=REPO_DIR, check=True)

## Install/check dependencies

In [ ]:
# Kaggle images may not include segmentation_models_pytorch by default.
try:
    import segmentation_models_pytorch as smp
    import cv2
    import shapely
    print('segmentation_models_pytorch:', smp.__version__)
except Exception as exc:
    print('Installing missing requirements because import failed:', repr(exc))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SEG_DIR / 'requirements.txt')], check=True)
    import segmentation_models_pytorch as smp
    print('segmentation_models_pytorch:', smp.__version__)

## Resolve dataset path

In [ ]:
def valid_data_root(path: Path) -> bool:
    return (path / 'metadata.csv').exists() and (path / 'images').is_dir() and (path / 'masks').is_dir()

# Prefer the current project Kaggle dataset. Keep env override for re-runs.
candidates = []
if os.environ.get('DATA_ROOT'):
    candidates.append(Path(os.environ['DATA_ROOT']))
candidates.extend([
    Path('/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset/segmentation_dataset'),
    Path('/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset'),
    Path('/kaggle/input/kurgans-dataset/segmentation_dataset/segmentation_dataset'),
    Path('/kaggle/input/kurgans-dataset/segmentation_dataset'),
])

DATA_ROOT = next((p for p in candidates if valid_data_root(p)), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find dataset root. Set DATA_ROOT to a folder with metadata.csv/images/masks.')

RUN_ROOT = Path(os.environ.get('RUN_ROOT', str(SEG_DIR / 'runs')))
OUT_DIR = RUN_ROOT / 'multiclass' / 'archaeology_5class_old_baseline_resnet34'
SMOKE_OUT_DIR = RUN_ROOT / 'multiclass' / 'archaeology_5class_old_baseline_smoke'

print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('OUT_DIR:', OUT_DIR)

## Sanity check metadata and old validation regions

Expected old-baseline split after filtering was `train=2656`, `val=223` on the old dataset copy. On `kurgans-dataset/segmentation_dataset`, `089_Костомаровское` has no samples after filtering, so the clean config omits it from the effective validation-region list.

In [ ]:
import pandas as pd
from IPython.display import display

OLD_VAL_REGIONS = [
    '005_ЛУБНО',
    '012_ЛИХУША',
    '008_СЕЛЯНЕ',
    '011_РУНА',
    '014_СТРЕКАЛОВКА',
    '016_ЗОЛОТАРЕВКА',
    '044_ГОЧЕВО',
]
VAL_REGIONS = ','.join(OLD_VAL_REGIONS)

meta = pd.read_csv(DATA_ROOT / 'metadata.csv', dtype={'sample_id': str})
print('raw rows:', len(meta))
print('columns:', list(meta.columns))

if 'class_name' in meta.columns:
    print('\nclass_name counts')
    display(meta['class_name'].value_counts())
if 'modality' in meta.columns:
    print('\nmodality counts')
    display(meta['modality'].value_counts())

present_regions = set(meta['region'].astype(str))
missing_regions = [region for region in OLD_VAL_REGIONS if region not in present_regions]
if missing_regions:
    raise ValueError(f'Old validation regions are absent from metadata: {missing_regions}')

print('\nold val region counts')
display(meta[meta['region'].isin(OLD_VAL_REGIONS)]['region'].value_counts())

## Old recipe config sanity check

In [ ]:
import yaml

config_path = SEG_DIR / 'configs' / 'archaeology_5class_old_baseline.yaml'
assert config_path.exists(), config_path
config = yaml.safe_load(config_path.read_text())
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))

## Smoke run

Default: run a 2-epoch smoke test first. Set `RUN_SMOKE=0` in notebook environment to skip.

In [ ]:
RUN_SMOKE = os.environ.get('RUN_SMOKE', '1') == '1'
if RUN_SMOKE:
    subprocess.run([
        sys.executable, 'scripts/train.py',
        '--config', 'configs/archaeology_5class_old_baseline.yaml',
        '--data-root', str(DATA_ROOT),
        '--out-dir', str(SMOKE_OUT_DIR),
        '--epochs', '2',
        '--batch-size', '2',
        '--num-workers', '0',
        '--save-samples', '2',
    ], cwd=SEG_DIR, check=True)
else:
    print('Smoke skipped')

## Full old-baseline reproduction training

Default: run full training. Set `RUN_TRAIN=0` if you only want to inspect setup or evaluate an already existing checkpoint.

In [ ]:
RUN_TRAIN = os.environ.get('RUN_TRAIN', '1') == '1'
if RUN_TRAIN:
    subprocess.run([
        sys.executable, 'scripts/train.py',
        '--config', 'configs/archaeology_5class_old_baseline.yaml',
        '--data-root', str(DATA_ROOT),
        '--out-dir', str(OUT_DIR),
    ], cwd=SEG_DIR, check=True)
else:
    print('Training skipped')

## Evaluate pixel and object metrics

In [ ]:
checkpoint = OUT_DIR / 'best_model.pth'
if checkpoint.exists():
    subprocess.run([
        sys.executable, 'scripts/evaluate.py',
        '--checkpoint', str(checkpoint),
        '--data-root', str(DATA_ROOT),
        '--out-dir', str(OUT_DIR),
        '--eval-mode', 'pixel',
    ], cwd=SEG_DIR, check=True)
    subprocess.run([
        sys.executable, 'scripts/evaluate.py',
        '--checkpoint', str(checkpoint),
        '--data-root', str(DATA_ROOT),
        '--out-dir', str(OUT_DIR),
        '--eval-mode', 'object',
        '--object-iou-threshold', '0.3',
        '--min-component-area', '8',
    ], cwd=SEG_DIR, check=True)
else:
    print('Checkpoint not found yet:', checkpoint)

## Visualize predictions

In [ ]:
if checkpoint.exists():
    subprocess.run([
        sys.executable, 'scripts/visualize_predictions.py',
        '--checkpoint', str(checkpoint),
        '--data-root', str(DATA_ROOT),
        '--output', str(OUT_DIR / 'prediction_examples.png'),
        '--max-samples', '8',
    ], cwd=SEG_DIR, check=True)
else:
    print('Checkpoint not found yet:', checkpoint)

## Results and quick comparison

Reference from old notebook evaluation:

- `weighted_competition_f1 ~= 0.741`
- reference checkpoint: `deeplab_5class_43_best.pth`

This notebook trains/evaluates the old recipe on the current `kurgans-dataset/segmentation_dataset`; use the resulting pixel/object metrics to see whether the recipe itself reproduces the old behavior.

In [ ]:
from IPython.display import display, Image
import json

for csv_name in ['history.csv', 'evaluation.csv', 'evaluation_object.csv', 'competition_metric.csv']:
    path = OUT_DIR / csv_name
    if path.exists():
        print('\n' + csv_name)
        df = pd.read_csv(path)
        display(df.tail() if len(df) > 5 else df)

summary_path = OUT_DIR / 'summary.json'
if summary_path.exists():
    print('\nsummary.json')
    print(json.dumps(json.loads(summary_path.read_text()), ensure_ascii=False, indent=2)[:3000])

image_path = OUT_DIR / 'prediction_examples.png'
if image_path.exists():
    display(Image(filename=str(image_path)))

## Zip outputs

In [ ]:
zip_path = Path('/kaggle/working/deeplab_old_baseline_reproduction_runs.zip')
if OUT_DIR.exists():
    subprocess.run(['zip', '-r', str(zip_path), str(OUT_DIR)], check=True)
    print('Saved:', zip_path)
else:
    print('Nothing to zip yet:', OUT_DIR)